# Quantum-limited imaging using diffractive optical neural networks — Fig. 1

Companion notebook for

> A. Warke, A. Zhang, A. I. Lvovsky, *Quantum-limited imaging using diffractive optical neural networks*, [arXiv:2608.12300](https://arxiv.org/abs/2608.12300).

**Fig. 1 — single-parameter estimation.** Per-photon variance of a single Fourier-cosine amplitude $a_m$ as a function of the normalized spatial frequency $k_\mathrm{obj}/(2\pi\,\mathrm{NA}/\lambda)$, comparing

- **direct imaging** — numerical CRB, its analytic envelope $2a_0^2/\mathrm{OTF}^2$, and the Monte-Carlo variance of the matched linear estimator;
- **quantum bounds** — QCRB, and the NHCRB computed by semidefinite programming (for a single parameter, NHCRB $=$ QCRB);
- **DONN** — the CRB of the trained phase-mask cascade and its Monte-Carlo variance, which saturate the quantum limit across the whole band.

**Outputs:** `figures/WZL_Fig1.png` / `.svg` and `data/WZL_Fig1.npz`.

**Requirements:** `numpy`, `torch`, `matplotlib`, `cvxpy` (Clarabel solver, for the NHCRB points). A CUDA GPU is used automatically if available; on consumer cards (FP64 at 1:64) set `USE_FP32 = True` in the configuration cell. Runtime is dominated by the batched DONN training in the cell marked **[slow]**; to regenerate the figure from previously saved data instead, run all cells *except* that one — the final **[fast]** cell re-plots from `data/WZL_Fig1.npz`.

In [ ]:
%matplotlib inline
import time
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
from pathlib import Path

try:
    import cvxpy as cp
except ImportError:
    cp = None
    print("cvxpy not found -> NHCRB points will be skipped (returned as inf)")

np.random.seed(0)
torch.set_default_dtype(torch.float64)
torch.backends.cuda.matmul.allow_tf32 = False   # TF32 would poison the dw@Pd contraction
torch.backends.cudnn.allow_tf32 = False

## Optics: 1D incoherent imaging model

`optics_exact_1d` builds the band-limited imaging operator on an $N$-point object grid of extent $L$ (units of $\lambda/\mathrm{NA}$-scaled coordinates): the Gram matrix $G_{jk}=\mathrm{sinc}\,[2\nu_c(x_j-x_k)]$ of the coherent PSFs is factorised as $G=TT^\top$ to obtain the mode structure of the image field, and $\psi_j(x)$ are the detector-plane amplitudes of a point source at $x_j$ (detector span $2L$). `cos_derivs` parametrises the (normalised) object intensity by Fourier-cosine amplitudes and returns $w=I(x)\,dx$ and $\partial w/\partial a_m$; `otf_triangle` is the incoherent OTF; `eigenbasis` projects $\rho$ and $\partial_a\rho$ onto the truncated eigenbasis used by the quantum bounds.

In [ ]:
def optics_exact_1d(NA=1.4, lam=0.540, L=10.8, N=256, det_span=2, tol=1e-12,
                    normalize_det=False):
    dx = L / N
    nc = NA / lam
    x_mid = (np.arange(N) + 0.5) * dx
    G = np.sinc(2 * nc * (x_mid[:, None] - x_mid[None, :]))
    ev, V = np.linalg.eigh(G); keep = ev > tol * ev.max()
    T = V[:, keep] * np.sqrt(ev[keep])
    Nd = det_span * N
    x_det = (np.arange(Nd) + 0.5) * dx - (Nd - N) // 2 * dx
    psi = np.sqrt(2 * nc) * np.sinc(2 * nc * (x_det[None, :] - x_mid[:, None])) * np.sqrt(dx)
    P_wf = psi ** 2
    capture = P_wf.sum(1)
    if normalize_det:
        psi = psi / np.sqrt(capture)[:, None]; P_wf = psi ** 2
    return dict(x_mid=x_mid, x_det=x_det, T=T, K=int(keep.sum()), N=N, Nd=Nd,
                L=L, dx=dx, nu_coh=nc, nu_incoh=2 * nc, capture=capture,
                M_max=int(np.floor(4 * NA * L / lam)), psi=psi, P_wf=P_wf)


def cos_derivs(m_list, opt, a0=1.0):
    L, N, dx, x = opt['L'], opt['N'], opt['dx'], opt['x_mid']
    ft = a0 * N * dx; w = np.full(N, a0) / ft * dx
    dw = np.zeros((len(m_list), N))
    for i, m in enumerate(m_list):
        C = np.cos(np.pi * m / L * x); dft = np.sum(C) * dx
        dw[i] = (C * ft - a0 * dft) / ft ** 2 * dx
    return w, dw


def otf_triangle(nu, opt):
    return np.clip(1.0 - nu / opt['nu_incoh'], 0.0, 1.0)


def eigenbasis(w, dw_list, opt):
    T = opt['T']
    rho = T.T @ (w[:, None] * T)
    drf = [T.T @ (dw[:, None] * T) for dw in dw_list]
    ev, U = np.linalg.eigh(rho); keep = ev > 1e-10 * ev.max()
    lamk, Uk = ev[keep], U[:, keep]
    return np.diag(lamk), [Uk.T @ dr @ Uk for dr in drf], len(lamk)

## Precision bounds

- `crb_di` — classical CRB of ideal photon counting on the image plane (direct imaging): $\mathrm{Tr}\,F^{-1}$ with the Poisson Fisher information $F_{ab}=\sum_d \partial_a u_d\,\partial_b u_d/u_d$.
- `crb_qfi` — quantum CRB from the SLD quantum Fisher information evaluated in the eigenbasis of $\rho$.
- `crb_nhcrb` — Nagaoka–Hayashi CRB via its semidefinite-programming form (solved with Clarabel), with per-parameter rescaling of $\partial_a\rho$ for numerical conditioning.

In [ ]:
def crb_di(w, dw_list, opt, Nph=1.0):
    P = opt['P_wf']; M = len(dw_list)
    u = Nph * (w[:, None] * P).sum(0); du = [Nph * (dw[:, None] * P).sum(0) for dw in dw_list]
    F = np.array([[np.sum(du[a] * du[b] / (u + 1e-15)) for b in range(M)] for a in range(M)])
    return np.trace(np.linalg.inv(F))


def crb_qfi(rho_eig, dr_list, Nph=1.0):
    lam = np.diag(rho_eig); M = len(dr_list)
    den = lam[:, None] + lam[None, :]; ok = den > 1e-12 * lam.max()
    inv = np.zeros_like(den); inv[ok] = 1.0 / den[ok]
    Q = np.array([[Nph * 2 * np.sum(dr_list[a] * dr_list[b] * inv).real for b in range(M)]
                  for a in range(M)])
    return np.trace(np.linalg.inv(Q))


def crb_nhcrb(rho_eig, dr_list, K, Nph=1.0, solver=None, max_iters=60000, eps=1e-9):
    if cp is None:
        return np.inf
    solver = solver or cp.CLARABEL
    M = len(dr_list)
    if K < 2: return np.inf
    dn = np.array([np.linalg.norm(dd) for dd in dr_list])
    drt = [dd / nn for dd, nn in zip(dr_list, dn)]
    wgt = 1.0 / dn ** 2
    Xs = [cp.Variable((K, K), symmetric=True) for _ in range(M)]
    Ls = {(j, k): cp.Variable((K, K), symmetric=True) for j in range(M) for k in range(j, M)}
    gL = lambda j, k: Ls[(j, k)] if j <= k else Ls[(k, j)].T
    cons = [cp.trace(drt[j] @ Xs[k]) == (1 if j == k else 0) for j in range(M) for k in range(M)]
    cons.append(cp.bmat([[gL(j, k) for k in range(M)] + [Xs[j]] for j in range(M)]
                        + [[Xs[m].T for m in range(M)] + [np.eye(K)]]) >> 0)
    prob = cp.Problem(cp.Minimize(sum(wgt[j] * cp.trace(rho_eig @ Ls[(j, j)]) for j in range(M))), cons)
    try:
        kw = {} if solver == cp.CLARABEL else dict(max_iters=max_iters, eps=eps)
        prob.solve(solver=solver, verbose=False, **kw)
        if prob.value is not None and np.isfinite(prob.value): return prob.value / Nph
    except Exception:
        pass
    return np.inf

## DONN: batched multi-plane light conversion

`MPLCBatch` propagates the source amplitudes through `n_planes` trainable phase masks separated by optical Fourier transforms. All $B$ single-parameter problems (one per spatial frequency $m$) are trained **simultaneously** as independent entries of one batch — the losses never mix, so one backward pass is exactly equivalent to $B$ separate runs, but saturates the GPU. The loss per entry is $\mathrm{Tr}\,F^{-1}=1/F$ (single parameter, so no matrix inverse). `donn_planes_batch` trains over a plane ladder (2 → 4 → 8) with warm starts and returns, per frequency, the best masks seen at any epoch of any rung. `donn_detector` converts trained masks into detection probabilities $|U\psi_j|^2$ for the Monte-Carlo runs.

In [ ]:
def pick_device(arg=None):
    if arg: return torch.device(arg)
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class MPLCBatch(nn.Module):

    def __init__(self, opt, m_list, n_planes, a0=1.0, init=None,
                 device='cpu', cdtype=torch.complex128, gen=None):
        super().__init__()
        Nd = opt['psi'].shape[1]; B = len(m_list)
        self.n, self.B, self.Nd = n_planes, B, Nd
        self.cdtype = cdtype
        rdtype = torch.float64 if cdtype == torch.complex128 else torch.float32
        self.register_buffer('psi', torch.as_tensor(opt['psi'])
                             .to(device=device, dtype=cdtype).unsqueeze(0))  # (1,Ns,Nd)
        w, dw = cos_derivs(m_list, opt, a0)                                   # (Ns,), (B,Ns)
        self.register_buffer('w', torch.as_tensor(w).to(device=device, dtype=torch.float64))
        self.register_buffer('dw', torch.as_tensor(dw).to(device=device, dtype=torch.float64))
        if init is None:
            init = [torch.randn(B, Nd, generator=gen, dtype=torch.float64) * 0.5
                    for _ in range(n_planes)]
        self.masks = nn.ParameterList(
            [nn.Parameter(p.detach().clone().to(device=device, dtype=rdtype)) for p in init])

    def _U(self, psi):
        for k in range(self.n):
            psi = psi * torch.exp(1j * self.masks[k].to(psi.dtype)).unsqueeze(1)
            psi = torch.fft.fftshift(
                torch.fft.fft(torch.fft.ifftshift(psi, dim=-1), norm='ortho'), dim=-1)
        return psi

    def losses(self):
        """Per-batch-entry Tr(F^-1); shape (B,).  Fisher assembled in fp64 always."""
        psi = self._U(self.psi)                                    # (B,Ns,Nd)
        Pd = (psi.real ** 2 + psi.imag ** 2).to(torch.float64)     # 2.3x cheaper than abs()**2
        u = torch.einsum('j,bjd->bd', self.w, Pd)                  # (B,Nd)
        du = torch.einsum('bj,bjd->bd', self.dw, Pd)               # (B,Nd)
        floor = 1e-15 * u.amax(dim=1, keepdim=True)                # relative, fp32-safe
        F = (du * du / (u + floor)).sum(1)                         # (B,)  M=1 -> no inv()
        return 1.0 / F

    def forward(self):
        return self.losses().sum()


def _train_batch(model, epochs, lr):
    dev = model.masks[0].device
    optm = optim.Adam(model.parameters(), lr=lr, foreach=True)
    best = torch.full((model.B,), float('inf'), dtype=torch.float64, device=dev)
    bm = [p.detach().clone() for p in model.masks]
    for _ in range(epochs):
        optm.zero_grad(set_to_none=True)
        v = model.losses()
        v.sum().backward()
        # snapshot BEFORE stepping, so the saved masks are the ones that produced v.
        with torch.no_grad():
            better = v < best
            torch.where(better, v, best, out=best)
            sel = better.unsqueeze(1)
            for b, p in zip(bm, model.masks):
                torch.where(sel, p.detach(), b, out=b)
        optm.step()
    return best, bm


def donn_planes_batch(opt, m_list, ladder=(2, 4, 8), epochs=1500, lr=0.04, lr_decay=0.5,
                      jitter=1e-3, device='cpu', cdtype=torch.complex128, seed=0,
                      verbose=False):
    gen = torch.Generator().manual_seed(seed)          # CPU generator -> reproducible on any device
    Nd = opt['psi'].shape[1]; B = len(m_list)
    best_val = torch.full((B,), float('inf'), dtype=torch.float64)
    best_stage = torch.zeros(B, dtype=torch.long)
    stage_masks, prev, cur_lr = [], None, lr
    for s, P in enumerate(ladder):
        if prev is None:
            init = None
        else:
            if len(prev) > P:
                raise ValueError("ladder must be non-decreasing")
            extra = [torch.randn(B, Nd, generator=gen, dtype=torch.float64) * jitter
                     for _ in range(P - len(prev))]
            init = list(prev) + [e.to(prev[0].device) for e in extra]
        model = MPLCBatch(opt, m_list, P, init=init, device=device, cdtype=cdtype, gen=gen)
        v, mk = _train_batch(model, epochs, cur_lr)
        v = v.cpu()                                     # one sync per rung, not per epoch
        stage_masks.append([m.detach().cpu().clone() for m in mk])
        better = v < best_val
        best_stage = torch.where(better, torch.full_like(best_stage, s), best_stage)
        best_val = torch.where(better, v, best_val)
        if verbose:
            print(f"    {P:3d} planes -> mean Tr(F^-1) = {v.mean():.4f}   worst {v.max():.4f}")
        prev = mk
        cur_lr *= lr_decay
    out = [[stage_masks[best_stage[b]][k][b].clone()
            for k in range(len(stage_masks[best_stage[b]]))] for b in range(B)]
    return best_val.numpy(), out


def donn_detector(opt, masks):
    
    dev = masks[0].device
    psi = torch.as_tensor(opt['psi'], dtype=torch.complex128, device=dev)
    for mk in masks:
        psi = psi * torch.exp(1j * mk.to(torch.complex128)).unsqueeze(0)
        psi = torch.fft.fftshift(
            torch.fft.fft(torch.fft.ifftshift(psi, dim=1), norm='ortho'), dim=1)
    return (psi.real ** 2 + psi.imag ** 2).double().cpu().numpy()

### Optional: benchmark batched vs looped training

In [ ]:
def bench(opt, m_list, device, cdtype, epochs=60, planes=4):
   
    def sync():
        if device.type == 'cuda': torch.cuda.synchronize()

    mb = MPLCBatch(opt, m_list, planes, device=device, cdtype=cdtype)
    for _ in range(5):
        mb.zero_grad(set_to_none=True); mb.losses().sum().backward()
    sync(); t = time.time()
    for _ in range(epochs):
        mb.zero_grad(set_to_none=True); mb.losses().sum().backward()
    sync(); t_b = (time.time() - t) / epochs * 1e3

    ms = [MPLCBatch(opt, [m], planes, device=device, cdtype=cdtype) for m in m_list]
    for m in ms[:2]:
        m.zero_grad(set_to_none=True); m.losses().sum().backward()
    sync(); t = time.time()
    for _ in range(epochs):
        for m in ms:
            m.zero_grad(set_to_none=True)
            m.losses().sum().backward()
            m.losses().item()          # mimics a per-epoch host sync
    sync(); t_l = (time.time() - t) / epochs * 1e3

    print(f"  batched  : {t_b:8.2f} ms / epoch  (all {len(m_list)} m)")
    print(f"  looped   : {t_l:8.2f} ms / epoch  (all {len(m_list)} m)")
    print(f"  speedup  : x{t_l/t_b:.2f}")
    return t_l / t_b

## Monte-Carlo validation

Poisson shot noise at $N=10^5$ photons per trial, $T=5000$ trials. The estimator is the matched (inverse-variance-weighted) linear estimator built from the detector derivatives; `mc_pervar` returns $N\cdot\mathrm{Tr\,cov}$, i.e. the per-photon variance to compare directly with the bounds.

In [ ]:
def matched_filter(n, D, u0, N):
    F = (D / np.sqrt(u0)) @ (D / np.sqrt(u0)).T
    return np.linalg.solve(F, D @ (n / u0)) / N


def mc_pervar(w, dw, P, N=100000, T=5000, seed=0):
    u0 = w @ P; D = dw @ P
    F = (D / np.sqrt(u0)) @ (D / np.sqrt(u0)).T; Finv = np.linalg.inv(F)
    rng = np.random.default_rng(seed)
    n = rng.poisson(N * u0, size=(T, len(u0)))
    est = ((n / u0) @ D.T) @ Finv.T / N
    cov = np.cov(est.T) if D.shape[0] > 1 else np.array([[est[:, 0].var(ddof=1)]])
    return np.trace(cov) * N

## Sweep over spatial frequencies

Dense numerical DI-CRB and QCRB curves over all $m$ (cheap, CPU); batched DONN training for the 16 sparse $m$ values; then per $m$: NHCRB SDP, DONN CRB, and Monte-Carlo variances for direct imaging and the DONN.

In [ ]:
def sweep_single(opt, m_sparse, epochs=1500, ladder=(2, 4, 8), lr=0.04,
                 device='cpu', cdtype=torch.complex128, nh_iters=60000, nh_eps=1e-9,
                 mc_T=5000, mc_N=100000, seed=0, verbose=True):
    # dense numeric DI/QCRB curves (cheap, CPU)
    m_dense = list(range(1, opt['M_max'] - 1)); di_d, q_d = [], []
    for m in m_dense:
        w, dw = cos_derivs([m], opt); re, dr, K = eigenbasis(w, [dw[0]], opt)
        di_d.append(crb_di(w, [dw[0]], opt)); q_d.append(crb_qfi(re, dr))

    t0 = time.time()
    donn_vals, donn_masks = donn_planes_batch(opt, m_sparse, ladder=ladder, epochs=epochs,
                                              lr=lr, device=device, cdtype=cdtype,
                                              seed=seed, verbose=verbose)
    if verbose:
        print(f"  DONN training ({len(m_sparse)} m-values, batched): {time.time()-t0:.1f} s")

    nh, dn, di_n, q_n, mc_di, mc_dn = [], [], [], [], [], []
    if verbose:
        print(f"{'m':>4} {'nu/nuc':>7} {'DI_num':>8} {'DI_MC':>8} {'QCRB':>8} "
              f"{'DONN':>8} {'DONN_MC':>8} {'D/Q':>6}")
    for i, m in enumerate(m_sparse):
        w, dw = cos_derivs([m], opt); re, dr, K = eigenbasis(w, [dw[0]], opt)
        cdi, cq = crb_di(w, [dw[0]], opt), crb_qfi(re, dr)
        cn = crb_nhcrb(re, dr, K, max_iters=nh_iters, eps=nh_eps)
        b4 = float(donn_vals[i])
        v_di = mc_pervar(w, dw, opt['P_wf'], N=mc_N, T=mc_T)
        v_dn = mc_pervar(w, dw, donn_detector(opt, donn_masks[i]), N=mc_N, T=mc_T)
        nh.append(cn); dn.append(b4); di_n.append(cdi); q_n.append(cq)
        mc_di.append(v_di); mc_dn.append(v_dn)
        if verbose:
            print(f"{m:4d} {m/(2*opt['L'])/opt['nu_coh']:7.3f} {cdi:8.3f} {v_di:8.3f} "
                  f"{cq:8.3f} {b4:8.3f} {v_dn:8.3f} {b4/cq:6.3f}")
    return dict(m_sparse=m_sparse, nu_sparse=np.array([m / (2 * opt['L']) for m in m_sparse]),
                nhcrb=np.array(nh), donn4=np.array(dn), di_num=np.array(di_n),
                qcrb_num=np.array(q_n), mc_di=np.array(mc_di), mc_donn=np.array(mc_dn),
                mc_T=mc_T, nu_dense=np.array([m / (2 * opt['L']) for m in m_dense]),
                di_dense=np.array(di_d), qcrb_dense=np.array(q_d))

## Figure

In [ ]:
def make_plot(opt, res, fname="figures/WZL_Fig1.png"):
    xd = res["nu_dense"] / opt["nu_coh"]
    nu = np.linspace(1e-3, 1.97 * opt["nu_coh"], 400)
    o = otf_triangle(nu, opt)
    xn = nu / opt["nu_coh"]

    fig, ax = plt.subplots(figsize=(9, 6))
    xs = res["nu_sparse"] / opt["nu_coh"]

    ax.plot(xd, res["qcrb_dense"], color="crimson", lw=2, label=r"QCRB (numerical)")
    ax.plot(xn, 2 / o, color="crimson", lw=1, ls="--", alpha=0.5, label=r"2$a_0^2$/OTF")
    ax.plot(xd, res["di_dense"], color="0.4", lw=2, label=r"Direct imaging (numerical)")
    ax.plot(xn, 2 / o ** 2, color="0.4", lw=1, ls="--", alpha=0.5, label=r"2$a_0^2$/OTF$^2$")
    ax.plot(xs, res["nhcrb"], "s:", mfc="none", mec="C0", mew=1.8, ms=11, label=r"NHCRB$=$QCRB")
    ax.plot(xs, res["donn4"], "D", color="darkorange", ms=8, label="DONN CRB")
    ax.plot(xs, res["mc_di"], "x", color="k", ms=8, mew=1.6, label="Direct imaging (MC variance)")
    ax.plot(xs, res["mc_donn"], "+", color="k", ms=11, mew=1.6, label="DONN (MC variance)")

    ax.set_yscale("log"); ax.set_xlim(0, 2.05); ax.set_ylim(1.5, 1.2e3)
    y_text = ax.get_ylim()[1] * 0.6
    ax.axvline(1.0, color="0.3", ls=":", lw=1.2)
    ax.text(1.03, y_text, "coherent\ncutoff", ha="left", va="top", fontsize=14, color="0.3",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=2))
    ax.axvline(2.0, color="0.3", ls=":", lw=1.2)
    ax.text(1.97, y_text, "incoherent\ncutoff", ha="right", va="top", fontsize=14, color="0.3",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=2))

    ax.set_xlabel(r'Normalized spatial frequency $\left(\frac{k_{obj}}{2\pi\,\mathrm{NA}/\lambda}\right)$',
                  fontsize=16)
    ax.set_ylabel(r"Per-photon variance", fontsize=16)
    ax.tick_params(axis="both", which="major", labelsize=16)
    ax.legend(fontsize=14, loc="upper left")
    ax.grid(True, which="both", ls='-.', alpha=0.3)
    fig.tight_layout()
    out = Path(fname)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=500, bbox_inches="tight")
    fig.savefig(out.with_suffix(".svg"), bbox_inches="tight")
    print("saved", out, "and", out.with_suffix(".svg"))
    plt.show()

## Run

Configuration, device selection, and the optical model. With `RUN_BENCH = True` this cell also times batched vs looped training.

In [ ]:
# ---------------- configuration ----------------
DEVICE    = None      # None = auto-detect ('cuda' if available); or 'cuda' / 'cpu'
USE_FP32  = False     # True: propagate in complex64 (recommended on consumer GPUs, where FP64 runs at 1:64)
EPOCHS    = 1500      # Adam epochs per rung of the plane ladder
SEED      = 0
RUN_BENCH = False     # True: time batched vs looped training (optional)

FIG_DIR, DATA_DIR = Path("figures"), Path("data")

dev = pick_device(DEVICE)
cdtype = torch.complex64 if USE_FP32 else torch.complex128
if dev.type == 'cuda':
    name = torch.cuda.get_device_name(dev)
    cc = torch.cuda.get_device_capability(dev)
    print(f"device: {name}  (sm_{cc[0]}{cc[1]})  propagating in {cdtype}")
    if not USE_FP32 and ('GeForce' in name or 'RTX' in name):
        print("  ! consumer-class card: FP64 is typically 1:64 here, consider USE_FP32 = True")
else:
    print("device: cpu  (note: batching is a GPU optimisation; on CPU it is slower)")

opt = optics_exact_1d()
print(f"N={opt['N']} L={opt['L']} M_max={opt['M_max']} "
      f"nu_coh={opt['nu_coh']:.3f} nu_incoh={opt['nu_incoh']:.3f}")
m_sparse = [4, 12, 20, 28, 38, 48, 58, 68, 78, 86, 92, 97, 101, 104, 106, 108]

if RUN_BENCH:
    bench(opt, m_sparse, dev, cdtype)

### Full sweep — **[slow]**

Retrains all DONNs (batched, on the selected device), computes all bounds and Monte-Carlo variances, saves `data/WZL_Fig1.npz`, and draws the figure.

In [ ]:
res = sweep_single(opt, m_sparse, epochs=EPOCHS, device=dev, cdtype=cdtype, seed=SEED)

DATA_DIR.mkdir(exist_ok=True)
np.savez(DATA_DIR / "WZL_Fig1.npz", **res)
print("saved", DATA_DIR / "WZL_Fig1.npz")

make_plot(opt, res)

### Re-plot from saved data — **[fast]**

In [ ]:
# Fast path: regenerate the figure from the saved arrays
res_saved = dict(np.load(DATA_DIR / "WZL_Fig1.npz"))
make_plot(opt, res_saved)